# Transformer from scratch

This notebook builds a Transformer one piece at a time, in the same order as Lecture 8 of [CS231n](https://cs231n.stanford.edu/slides/2026/lecture_8.pdf). It continues the notebook `Self-Attention from Scratch`, which ends with the multi-headed self-attention layer. Here that layer becomes part of a block, the blocks are stacked, and the result is trained as a small language model.

| Step | Idea | Slides |
|---|---|---|
| 1 | Self-attention: the interaction between vectors | 85-86 |
| 2 | Residual connection | 87 |
| 3 | Layer normalization | 88 |
| 4 | MLP on each vector | 89 |
| 5 | The Transformer block | 90-92 |
| 6 | A Transformer is a stack of blocks | 93-96 |
| 7 | Transformers for language modeling | 97-100 |
| 8 | Training on a sequence task | |
| 9 | Generating one token at a time | |
| 10 | What did the attention learn? | |
| 11 | Removing the positional encoding | 59 |
| 12 | Pre-norm Transformer | 111-113 |

The task is the one from the notebook `Seq2Seq LSTM with PyTorch`: inverting a sequence. There, an encoder and a decoder had to pass the whole sequence through a single vector. Here, each output can look directly at the input it needs.

The model is small, so a CPU is enough, although a GPU is faster.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/08_transformers'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)
print(device)

## Step 1: self-attention, the interaction between vectors (slides 85-86)

A Transformer block receives a **set of vectors** $x$ and returns a set of vectors $y$. Inside the block, all vectors interact through (multi-headed) self-attention.

We start from the class built in step 14 of the previous notebook. `positional_encoding` also comes from there (step 9).

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, D, H):
        super().__init__()
        assert D % H == 0, "D must be divisible by H"
        self.H = H
        self.D_H = D // H
        self.qkv = nn.Linear(D, 3 * D)  # 1. QKV projection
        self.out = nn.Linear(D, D)  # 4. output projection

    def forward(self, X, mask=None):
        B, N, D = X.shape

        # 1. QKV projection, split and reshape
        Q, K, V = self.qkv(X).chunk(3, dim=2)  # each [B x N x D]
        Q = Q.reshape(B, N, self.H, self.D_H).transpose(1, 2)  # [B x H x N x D_H]
        K = K.reshape(B, N, self.H, self.D_H).transpose(1, 2)
        V = V.reshape(B, N, self.H, self.D_H).transpose(1, 2)

        # 2. QK similarity
        E = Q @ K.transpose(2, 3) / math.sqrt(self.D_H)  # [B x H x N x N]
        if mask is not None:
            E = E.masked_fill(mask, float("-inf"))
        A = torch.softmax(E, dim=3)

        # 3. V-weighting, and reshape
        Y = A @ V  # [B x H x N x D_H]
        Y = Y.transpose(1, 2).reshape(B, N, D)

        # 4. Output projection
        return self.out(Y), A


def positional_encoding(N, D):
    position = torch.arange(N, dtype=torch.float32)[:, None]  # [N x 1]
    frequency = 10000 ** (-torch.arange(0, D, 2, dtype=torch.float32) / D)  # [D/2]
    encoding = torch.zeros(N, D)
    encoding[:, 0::2] = torch.sin(position * frequency)
    encoding[:, 1::2] = torch.cos(position * frequency)
    return encoding


B, N, D, H = 2, 4, 16, 4
x = torch.randn(B, N, D)

attention = MultiHeadSelfAttention(D, H)
attended, _ = attention(x)
print(f"x: {tuple(x.shape)}, after self-attention: {tuple(attended.shape)}")

## Step 2: residual connection (slide 87)

The output of self-attention is added to its input: $x + \text{SelfAttention}(x)$. As in the ResNet, the block only has to learn a correction to the input, and the gradient has a direct path to the earlier layers. That is why the shapes have to match, and why self-attention returns $D$ dimensions.

In [ ]:
residual = x + attended
print(f"{tuple(residual.shape)}")

## Step 3: layer normalization (slide 88)

Layer normalization makes the training more stable. It normalizes **each vector on its own**, over its $D$ dimensions, and then applies a learned scale $\gamma$ and shift $\beta$:

$$\mu_i = \frac{1}{D}\sum_j h_{i,j} \qquad \sigma_i = \Big(\frac{1}{D}\sum_j (h_{i,j}-\mu_i)^2\Big)^{1/2} \qquad z_i = \frac{h_i - \mu_i}{\sigma_i} \qquad y_i = \gamma \odot z_i + \beta$$

Unlike batch normalization, it uses no statistics from other vectors. We write it by hand and compare with `nn.LayerNorm`:

In [ ]:
norm = nn.LayerNorm(D)  # starts with gamma = 1 and beta = 0

mean = residual.mean(dim=2, keepdim=True)
variance = residual.var(dim=2, keepdim=True, unbiased=False)
normalized = (residual - mean) / torch.sqrt(variance + norm.eps)

print(torch.allclose(normalized, norm(residual), atol=1e-5))
print(f"means of the first vectors: {normalized[0].mean(dim=1).detach()}")
print(f"stds of the first vectors:  {normalized[0].std(dim=1, unbiased=False).detach()}")

## Step 4: MLP on each vector (slide 89)

Next comes an MLP, usually with two layers: the hidden layer has $4D$ dimensions, and the output has $D$ again. It is applied to **each vector independently**, with the same weights.

`nn.Linear` already acts on the last dimension, so nothing special is needed. To confirm that the vectors don't interact, we change one input vector. Only the corresponding output should change:

In [ ]:
class MLP(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))

    def forward(self, x):
        return self.layers(x)


mlp = MLP(D)
y = mlp(x)

x_changed = x.clone()
x_changed[0, 2] = torch.randn(D)  # change vector 2 of the first sequence
y_changed = mlp(x_changed)

for i in range(N):
    print(f"vector {i}: output changed = {not torch.allclose(y[0, i], y_changed[0, i])}")

## Step 5: the Transformer block (slides 90-92)

The block joins these pieces: self-attention, residual, normalization, MLP, residual, normalization.

$$h = \text{LayerNorm}(x + \text{SelfAttention}(x)) \qquad y = \text{LayerNorm}(h + \text{MLP}(h))$$

Slide 92 sums up the properties of the block:

- Input and output are sets of vectors.
- **Self-attention is the only interaction between vectors.** LayerNorm and MLP work on each vector independently.
- Most of the compute is just **6 matrix multiplies**: 4 from self-attention and 2 from the MLP.

The `pre_norm` argument only matters in step 12.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, D, H, pre_norm=False):
        super().__init__()
        self.attention = MultiHeadSelfAttention(D, H)
        self.norm1 = nn.LayerNorm(D)
        self.mlp = MLP(D)
        self.norm2 = nn.LayerNorm(D)
        self.pre_norm = pre_norm

    def forward(self, x, mask=None):
        if self.pre_norm:
            attended, A = self.attention(self.norm1(x), mask)
            x = x + attended
            x = x + self.mlp(self.norm2(x))
        else:
            attended, A = self.attention(x, mask)
            x = self.norm1(x + attended)
            x = self.norm2(x + self.mlp(x))
        return x, A


block = TransformerBlock(D, H)
y, A = block(x)
print(f"x: {tuple(x.shape)}, y: {tuple(y.shape)}, A: {tuple(A.shape)}")

## Step 6: a Transformer is a stack of blocks (slides 93-96)

A Transformer is just a stack of identical blocks: the output of one block is the input of the next. Since 2017, the architecture has not changed much, but it has become a lot bigger.

Almost all the parameters are in the 6 matrices of each block: $4D^2$ in self-attention and $8D^2$ in the MLP, so about $12D^2$ per block. With this we can estimate the sizes shown on slide 96:

In [ ]:
print(f"{'Model':<12}{'Blocks':>8}{'D':>8}{'12 * L * D^2':>16}{'On the slide':>14}")
for name, L, D_model, reported in [
    ("Original", 12, 1024, "213M"),
    ("GPT-2", 48, 1600, "1.5B"),
    ("GPT-3", 96, 12288, "175B"),
]:
    estimate = 12 * L * D_model**2
    print(f"{name:<12}{L:>8}{D_model:>8}{estimate / 1e6:>14,.0f}M{reported:>14}")

block = TransformerBlock(1024, 16)
print(f"\nOne block with D=1024 has {sum(p.numel() for p in block.parameters()):,} parameters, and 12 * D^2 = {12 * 1024**2:,}")

The estimate matches for GPT-2 and GPT-3. For the original model it falls short because that model also has an embedding table, and a decoder with extra cross-attention layers.

## Step 7: Transformers for language modeling (slides 97-100)

A Transformer works on vectors, so we need to turn tokens into vectors and vectors back into tokens. A language model needs three additions:

1. **Embedding matrix** $[V \times D]$: a lookup table that converts each token into a vector, where $V$ is the vocabulary size.
2. **Masked attention** in each block, so that each token can only see the ones before it.
3. **Projection matrix** $[D \times V]$: converts each vector into a score for each token of the vocabulary.

The model is trained to predict the next token, using softmax and cross-entropy loss. We also add the positional encoding to the embeddings, since self-attention alone does not know the order.

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, V, D, H, L, max_length, use_positions=True, pre_norm=False):
        super().__init__()
        self.embedding = nn.Embedding(V, D)  # embedding matrix [V x D]
        self.blocks = nn.ModuleList([TransformerBlock(D, H, pre_norm) for _ in range(L)])
        self.final_norm = nn.LayerNorm(D) if pre_norm else nn.Identity()
        self.projection = nn.Linear(D, V)  # projection matrix [D x V]
        self.use_positions = use_positions
        self.register_buffer("positions", positional_encoding(max_length, D))

    def forward(self, tokens):
        B, N = tokens.shape
        x = self.embedding(tokens)  # [B x N x D]
        if self.use_positions:
            x = x + self.positions[:N]

        # Each token sees only itself and the tokens before it
        mask = torch.triu(torch.ones(N, N, dtype=torch.bool, device=tokens.device), diagonal=1)

        attentions = []
        for block in self.blocks:
            x, A = block(x, mask)
            attentions.append(A)

        scores = self.projection(self.final_norm(x))  # [B x N x V]
        return scores, attentions

## Step 8: training on a sequence task

**The task.** A language model only knows how to continue a sequence. So we write the input and the expected output as one sequence, separated by a token `>`, and let the model complete it:

```
3 7 1 9 > 9 1 7 3
```

To predict each token after `>`, the model has to find the right token in the first half. Each sequence has the same length, so no padding is needed.

**The loss.** The target of each position is the next token. The first half is random, so there is nothing to learn from it: `ignore_index` removes those positions from the loss.

In [ ]:
length = 8  # number of digits to invert
SEPARATOR = 10
V = 11  # the digits 0-9 and the separator
IGNORE = -100


def make_batch(batch_size):
    digits = torch.randint(0, 10, (batch_size, length))
    separator = torch.full((batch_size, 1), SEPARATOR)
    sequence = torch.cat([digits, separator, digits.flip(dims=[1])], dim=1)

    inputs = sequence[:, :-1]  # each position receives a token...
    targets = sequence[:, 1:].clone()  # ...and predicts the next one
    targets[:, : length - 1] = IGNORE  # no loss on the digits of the input
    return inputs.to(device), targets.to(device)


inputs, targets = make_batch(1)
print(f"inputs:  {inputs[0].tolist()}")
print(f"targets: {targets[0].tolist()}")

A standard training loop. We keep the settings in one function because steps 11 and 12 train other versions of the model.

In [ ]:
def train(model, steps=1500, batch_size=64, learning_rate=0.001):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    model.train()
    for step in range(1, steps + 1):
        inputs, targets = make_batch(batch_size)
        scores, _ = model(inputs)

        # cross_entropy expects [number of predictions x V]
        loss = F.cross_entropy(scores.reshape(-1, V), targets.reshape(-1), ignore_index=IGNORE)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append(loss.item())
        if step % 300 == 0:
            print(f"step {step}: loss {loss.item():.4f}")
    return history


def new_model(**kwargs):
    torch.manual_seed(42)
    return TransformerLM(V=V, D=64, H=4, L=2, max_length=2 * length + 1, **kwargs).to(device)


model = new_model()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
history = train(model)

plt.plot(history)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training loss")
plt.show()

## Step 9: generating one token at a time

To invert a new sequence, we give the model the digits and the separator. It predicts one token, we append this token to the sequence, and we repeat. This is how every language model generates text.

In [ ]:
@torch.no_grad()
def generate(model, digits):
    model.eval()
    separator = torch.full((digits.shape[0], 1), SEPARATOR, device=device)
    sequence = torch.cat([digits, separator], dim=1)
    for _ in range(length):
        scores, _ = model(sequence)
        next_token = scores[:, -1].argmax(dim=1, keepdim=True)  # the most likely next token
        sequence = torch.cat([sequence, next_token], dim=1)
    return sequence[:, length + 1 :]


def evaluate(model, samples=1000):
    digits = torch.randint(0, 10, (samples, length), device=device)
    generated = generate(model, digits)
    expected = digits.flip(dims=[1])
    return (generated == expected).float().mean().item(), (generated == expected).all(dim=1).float().mean().item()


digits = torch.randint(0, 10, (5, length), device=device)
for source, result in zip(digits, generate(model, digits)):
    print(f"{source.tolist()} -> {result.tolist()}")

token_accuracy, sequence_accuracy = evaluate(model)
print(f"Correct tokens: {token_accuracy:.1%}")
print(f"Correct sequences: {sequence_accuracy:.1%}")

## Step 10: what did the attention learn?

In the previous notebook, the attention matrices were random. Now they are trained, so we can look at them. Each row is a query, and it shows which positions the model looked at to make that prediction.

To invert the sequence, the query that predicts the first output has to look at the last digit, the next one at the second-to-last digit, and so on. We expect to see this as a **diagonal going in the opposite direction** in the lower-left part of the matrices of the last block. The upper-right part is empty because of the mask.

The first block has a different job: it prepares the vectors for the second one, so its pattern is harder to read.

In [ ]:
digits = torch.randint(0, 10, (1, length), device=device)
result = generate(model, digits)
sequence = torch.cat([digits, torch.full((1, 1), SEPARATOR, device=device), result], dim=1)[:, :-1]

model.eval()
with torch.no_grad():
    _, attentions = model(sequence)

labels = [">" if token == SEPARATOR else str(token) for token in sequence[0].tolist()]
ticks = range(len(labels))

for layer, A in enumerate(attentions):
    fig, axes = plt.subplots(1, A.shape[1], figsize=(14, 4))
    for head, ax in enumerate(axes):
        ax.imshow(A[0, head].cpu(), cmap="viridis", vmin=0, vmax=1)
        ax.set_xticks(ticks, labels, fontsize=7)
        ax.set_yticks(ticks, labels, fontsize=7)
        ax.set_title(f"Block {layer + 1}, head {head + 1}")
    plt.tight_layout()
    plt.show()

## Step 11: removing the positional encoding (slide 59)

Slide 59 says that self-attention does not know the order of the sequence. To invert a sequence, order is everything. Let's train the same model without the positional encoding:

In [ ]:
model_no_positions = new_model(use_positions=False)
history_no_positions = train(model_no_positions)

token_accuracy, sequence_accuracy = evaluate(model_no_positions)
print(f"Without positional encoding: {token_accuracy:.1%} correct tokens, {sequence_accuracy:.1%} correct sequences")

The model learns much more slowly, and with the same training it still makes many mistakes, as the curve in the next step also shows. But notice that it does better than guessing. Slide 59 refers to self-attention without a mask. With a mask, each position sees a different number of tokens, so the model can extract some notion of position from that. The positional encoding gives this information directly.

## Step 12: pre-norm Transformer (slides 111-113)

The architecture has changed little since 2017, but a few changes became standard (slide 111). The first one moves the layer normalization. In the original block, it sits **outside** the residual connections, so the model can't actually learn the identity function. In the **pre-norm** block, it goes inside, right before self-attention and the MLP, which makes the training more stable:

$$h = x + \text{SelfAttention}(\text{LayerNorm}(x)) \qquad y = h + \text{MLP}(\text{LayerNorm}(h))$$

The change is already in `TransformerBlock`. A last LayerNorm before the projection is common in this version, and `TransformerLM` adds it. Let's compare the two:

In [ ]:
model_pre_norm = new_model(pre_norm=True)
history_pre_norm = train(model_pre_norm)

token_accuracy, sequence_accuracy = evaluate(model_pre_norm)
print(f"Pre-norm: {token_accuracy:.1%} correct tokens, {sequence_accuracy:.1%} correct sequences")

plt.plot(history, label="Post-norm (original)")
plt.plot(history_pre_norm, label="Pre-norm")
plt.plot(history_no_positions, label="Post-norm, no positional encoding")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

In a model with only two blocks, the two curves nearly coincide. The gain in stability appears in deep models, which are hard to train.

The other changes of slides 114-122 follow the same idea of small adjustments to the block:

- **QK-Norm:** normalizes queries and keys before computing the similarities.
- **SwiGLU:** replaces the MLP with a gated version.
- **Mixture of Experts:** keeps several MLPs in each block and sends each token to only a few of them.

## Summary

The whole Transformer fits in a few pieces:

| Piece | What it does | Do vectors interact? |
|---|---|---|
| Embedding + positional encoding | tokens become vectors that carry their position | no |
| Self-attention | mixes information between vectors | **yes** |
| Residual connection | keeps the input and adds a correction | no |
| LayerNorm | normalizes each vector | no |
| MLP | transforms each vector | no |
| Projection | vectors become scores for the next token | no |

Slides 101-110 apply the same blocks to images: the image is broken into patches, each patch becomes a vector, and no mask is used. The notebook `Vision Transformer with Hugging Face` and the activity AP4 continue from there.

### Try it yourself

1. In `make_batch`, change the task: instead of inverting the digits, copy them (remove `.flip(dims=[1])`, and also in `evaluate`). What does the attention pattern look like now?
2. Change `L` and `H` in `new_model`. How few blocks and heads are enough for this task?
3. In step 11, train the model without positional encoding for 5000 steps (`train(model_no_positions, steps=5000)`). How far does it get? What does this say about the information carried by the mask?

### References

- [CS231n, Lecture 8: Attention and Transformers](https://cs231n.stanford.edu/slides/2026/lecture_8.pdf)
- A. Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762), NeurIPS 2017.
- [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/): implements the full encoder-decoder model of the paper, line by line.
- A. Karpathy, [nanoGPT](https://github.com/karpathy/nanoGPT): a decoder-only Transformer like the one built here, at a larger scale.